# Customer Churn Prediction in the Nigerian Telecom Sector

Nigeria's telecommunications industry is one of the fastest-growing in Africa, with over 200 million subscribers across major providers. Despite this growth, churn — the rate at which customers stop using a provider's services — remains a pressing business challenge. Providers like **MTN Nigeria**, **Airtel Nigeria**, **Glo**, and **9mobile** face fierce competition, and retaining existing customers is far more cost-effective than acquiring new ones.

According to industry reports, acquiring a new telecom customer can cost up to five times more than retaining an existing one. A churn rate reduction of even 5% can significantly boost profitability. This makes churn prediction not just an academic exercise but a genuine business priority.

As the data analyst on this project, your goal is to build machine learning models that can predict whether a customer is likely to churn, and determine which model performs better for deployment in a real-world context.

You have been provided with two datasets from a regional telecom analytics firm:

- `telecom_demographics.csv` contains customer demographic information:

| Variable             | Description                                      |
|----------------------|--------------------------------------------------|
| `customer_id`        | Unique identifier for each customer.             |
| `telecom_partner`    | The telecom provider associated with the customer (MTN, Airtel, Glo, 9mobile). |
| `gender`             | The gender of the customer.                      |
| `age`                | The age of the customer.                         |
| `state`              | The Nigerian state in which the customer is located. |
| `city`               | The city in which the customer is located.       |
| `pincode`            | The area code of the customer's location.        |
| `registration_event` | When the customer first registered with the telecom provider. |
| `num_dependents`     | The number of dependents (e.g., children) the customer has. |
| `estimated_salary`   | The customer's estimated monthly income in Naira. |

- `telecom_usage.csv` contains customer usage behaviour:

| Variable      | Description                                                  |
|---------------|--------------------------------------------------------------|
| `customer_id` | Unique identifier for each customer.                         |
| `calls_made`  | The number of calls made by the customer in the last 30 days. |
| `sms_sent`    | The number of SMS messages sent by the customer in the last 30 days. |
| `data_used`   | The amount of mobile data used by the customer (in MB).      |
| `churn`       | Binary variable: 1 = churned (left the provider), 0 = still active. |

> **Business context:** A churn prediction model that is deployed by a telecom company could be used to flag at-risk customers and trigger proactive retention campaigns — such as personalised data bonuses, loyalty rewards, or targeted customer care calls. The accuracy of the model directly affects how efficiently the business can allocate its retention budget.


---

# Task Overview

**Core Question:** Does Logistic Regression or a Random Forest Classifier produce a higher accuracy score in predicting customer churn for a Nigerian telecom provider?

---

## Task 1 — Load, Merge, and Explore

Load `telecom_demographics.csv` and `telecom_usage.csv` into separate DataFrames, then merge them into a single DataFrame named `churn_df` using the shared `customer_id` column.

After merging:
- Calculate the **proportion of customers who have churned** (i.e., the churn rate). Think about what this tells you from a business standpoint — is it a balanced or imbalanced dataset?
- Identify all **categorical variables** in `churn_df`. These will need to be handled before modelling.

> **Real-world note:** Churn datasets are often imbalanced — far more customers stay than leave. Be mindful of this as you interpret accuracy scores later.


In [26]:
# Task 1: Load, merge, and explore

#importing 
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report



In [27]:
# loading datatsets

telecom_demographics_ng_df = pd.read_csv("telecom_demographics_nigeria.csv")
telecom_usage_df = pd.read_csv("telecom_usage.csv")

# checking datasets
telecom_demographics_ng_df.head()
telecom_usage_df.head()
telecom_demographics_ng_df.info()
telecom_usage_df.info()

FileNotFoundError: [Errno 2] No such file or directory: 'telecom_demographics_nigeria.csv'

In [ ]:
# merging datasets(demographics and usage data) on customer_id
churn_df = telecom_demographics_ng_df.merge(telecom_usage_df, on="customer_id")

# exploring
churn_df.head()
churn_df.shape
churn_df.info()
churn_df.describe()


<class 'pandas.DataFrame'>
RangeIndex: 6500 entries, 0 to 6499
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   customer_id         6500 non-null   int64
 1   telecom_partner     6500 non-null   str  
 2   gender              6500 non-null   str  
 3   age                 6500 non-null   int64
 4   state               6500 non-null   str  
 5   city                6500 non-null   str  
 6   pincode             6500 non-null   int64
 7   registration_event  6500 non-null   str  
 8   num_dependents      6500 non-null   int64
 9   estimated_salary    6500 non-null   int64
 10  calls_made          6500 non-null   int64
 11  sms_sent            6500 non-null   int64
 12  data_used           6500 non-null   int64
 13  churn               6500 non-null   int64
dtypes: int64(9), str(5)
memory usage: 711.1 KB


,customer_id,age,pincode,num_dependents,estimated_salary,calls_made,sms_sent,data_used,churn
count,6500.000000,6500.000000,6500.000000,6500.000000,6.500000e+03,6500.000000,6500.000000,6500.000000,6500.000000
mean,121035.576923,46.108615,548955.907077,1.982308,6.446543e+05,49.789538,24.257846,5000.956308,0.200462
std,70353.990092,16.443712,259874.312026,1.404659,3.235299e+05,29.799221,14.650736,2940.611928,0.400377
min,47.000000,18.000000,100045.000000,0.000000,8.000000e+04,-10.000000,-5.000000,-969.000000,0.000000
25%,60125.750000,32.000000,321550.000000,1.000000,3.635325e+05,25.000000,12.000000,2493.750000,0.000000
50%,120470.500000,46.000000,550163.500000,2.000000,6.425630e+05,50.000000,25.000000,4975.500000,0.000000
75%,181420.750000,60.000000,775155.500000,3.000000,9.314992e+05,75.000000,37.000000,7504.250000,0.000000
max,243505.000000,74.000000,999740.000000,4.000000,1.200000e+06,108.000000,53.000000,10919.000000,1.000000


In [28]:
# calculating the churn rate

churn_rate = churn_df["churn"].mean()

# format to percentage (2 dp)
print(f"Churn rate: {churn_rate:.2%}")

Churn rate: 20.05%


In [29]:
# identifying categorical columns in churn_df as a list

categorical_cols = churn_df.select_dtypes(include=["object", "string"]).columns.tolist()
print("Categorical columns:", categorical_cols)



Categorical columns: ['telecom_partner', 'gender', 'state', 'city', 'registration_event']


---

## Task 2 — Preprocess Features

Prepare the data for machine learning by doing the following:

1. **Encode categorical features** — Convert all categorical columns (e.g., `telecom_partner`, `gender`, `state`) into numeric format using one-hot encoding or label encoding.
2. **Select features** — Drop columns that would not be useful for prediction (e.g., `customer_id`, `city`, `pincode`, `registration_event`).
3. **Scale numeric features** — Apply feature scaling to the numerical columns. Store the result in a DataFrame called `features_scaled`.
4. **Define your target** — Isolate the `churn` column as your target variable `y`.

> **Why does scaling matter?** Logistic Regression is sensitive to feature magnitude — a salary in the millions of Naira and an age in the tens would otherwise be weighted very differently. Random Forest is less affected, but scaling is still good practice for fair comparison.


In [30]:
# Task 2: Encode, select features, scale, and define target

# dataset copy
model_df = churn_df.copy()

# unneeded columns
columns_drop = ["customer_id", "city", "pincode", "registration_event"]
model_df = model_df.drop(columns=columns_drop)

# Encode categorical columns with one encoding
model_df = pd.get_dummies(model_df, drop_first=True)

# Separate features
X = model_df.drop("churn", axis=1)

# Scale numerical features
scaler = StandardScaler()
features_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns
)

# isolate churn column as y
y = model_df["churn"]

# exploring
print("Feature matrix shape:", features_scaled.shape)
print("Target shape:", y.shape)
print(features_scaled)
print(features_scaled.var())



Feature matrix shape: (6500, 37)
Target shape: (6500,)
           age  num_dependents  estimated_salary  calls_made  sms_sent  \
0    -1.222970        1.436539          0.011980    0.846076 -0.222385   
1     1.696304       -1.411346         -0.428424   -0.496344  0.938056   
2     0.479940        0.012596         -0.255181    0.678273  1.552407   
3    -1.040515        0.724568         -1.365303    1.517286  0.528489   
4    -0.067424        1.436539         -1.368366    0.544031 -0.085862   
...        ...             ...               ...         ...       ...   
6495  0.479940        1.436539          1.046162   -1.738083  1.006317   
6496  1.392213       -0.699375         -0.530707   -0.999752  0.460228   
6497 -1.648697       -1.411346         -1.515530    0.778955 -0.700213   
6498 -1.222970        0.724568          0.911704    0.074184 -1.109780   
6499 -1.648697        0.724568          0.901339    1.349483 -0.836735   

      data_used  telecom_partner_Airtel Nigeria  telecom

---

## Task 3 — Train Models

Split your data into training and testing sets, then train two classification models:

1. **Split the data** into `X_train`, `X_test`, `y_train`, and `y_test` using an **80/20 split**, with `random_state=42` for reproducibility.
2. **Train a Logistic Regression model** — Use `random_state=42`. Store its predictions on the test set in `logreg_pred`.
3. **Train a Random Forest Classifier** — Use `random_state=42`. Store its predictions on the test set in `rf_pred`.

> **Real-world note:** In production, models are re-trained periodically on fresh data. The 80/20 split simulates this: training on historical data and testing on "unseen" new customers. Setting a random state ensures your results are reproducible — an important requirement in professional data science workflows.


In [31]:
# Task 3: Train-test split and model training

# Spliting dataset (training and testing)
X_train, X_test, y_train, y_test = train_test_split(
    features_scaled,
    y,
    test_size=0.2,
    random_state=42
)

# Training Logistic Regression model with rs = 42
logreg = LogisticRegression(random_state=42)
logreg.fit(X_train, y_train)

# Logistic regression prediction
logreg_pred = logreg.predict(X_test)

# Train Random Forest model, rs=42
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

# rf prediction
rf_pred = rf.predict(X_test)

# exploring
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (5200, 37)
X_test shape: (1300, 37)
y_train shape: (5200,)
y_test shape: (1300,)


---

## Task 4 — Evaluate and Compare Models

Assess both models on the test data:

1. **Calculate accuracy scores** for both `logreg_pred` and `rf_pred` against `y_test`.
2. **Print a classification report** for each model to understand precision, recall, and F1-score — these give a fuller picture than accuracy alone.
3. **Assign the name of the better-performing model** (by accuracy) to a variable called `higher_accuracy`. Use exactly one of these strings: `"LogisticRegression"` or `"RandomForest"`.

> **Reflection question (no code needed):** If you were advising the telecom company on which model to deploy, would accuracy alone be enough to make the decision? Consider: what is the cost of a *false negative* (predicting a customer won't churn when they actually will)?


In [32]:
# Task 4: Model evaluation

# calculating accuracy score 
logreg_pred_as = accuracy_score(y_test, logreg_pred)
rf_pred_as = accuracy_score(y_test, rf_pred)

print("Logistic Regression Accuracy:", logreg_pred_as)
print("Random Forest Accuracy:", rf_pred_as)

# classification report
print("\nLogistic Regression Classification Report:")
print(classification_report(y_test, logreg_pred, zero_division=0))

print("\nRandom Forest Classification Report:")
print(classification_report(y_test, rf_pred, zero_division=0))

# checking better performing model by accuracy
if logreg_pred_as > rf_pred_as:
    higher_accuracy = "LogisticRegression"
else:
    higher_accuracy = "RandomForest"

print("Higher accuracy model:", higher_accuracy)


Logistic Regression Accuracy: 0.79
Random Forest Accuracy: 0.7884615384615384

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.79      1.00      0.88      1027
           1       0.00      0.00      0.00       273

    accuracy                           0.79      1300
   macro avg       0.40      0.50      0.44      1300
weighted avg       0.62      0.79      0.70      1300


Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.79      1.00      0.88      1027
           1       0.00      0.00      0.00       273

    accuracy                           0.79      1300
   macro avg       0.39      0.50      0.44      1300
weighted avg       0.62      0.79      0.70      1300

Higher accuracy model: LogisticRegression


We can see that accuracy alone is not enough to determine whether the model is suitable for deployment because the dataset is imbalanced. About 80% of the customers did not churn, while only about 20% churned. This means a model can achieve a relatively high accuracy if predicting the majority class, which is non-churned customers in this case.
So, false negatives are significant in this project as a false negative prediction would indicate that the customer will not churn, whereas in reality, the client would leave. Therefore, failing to predict this would mean that there were no measures taken to retain the customer as required.
We need to use other evaluation metrics such as precision, recall, and F1-score are important to see how well the models predict the minority classes. From the classification reports printed, both Logistic Regression and Random Forest failed to identify churned customers, as the recall and F1-score for class 1 are 0.00. This shows that the models are mainly predicting the majority class. So, the model should be improved to increase recall and F1-score for class 1 so that more actual churners can be detected in a real company.